In [1]:
import os
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")
import numpy as np
import pandas as pd

df = pd.read_parquet("data/processed/model_table.parquet")
y = df["default_flag"]
x = df.drop(columns=["id", "default_flag", "issue_year", "issue_date", "funded_amnt", "annual_inc"])
x = pd.get_dummies(x, columns=["home_ownership", "verification_status", "purpose"], drop_first=True, dtype=int)

print(f"Shape of x: {x.shape}")
print(f"Mean of y: {y.mean()}")

train_mask = df["issue_year"] <= 2014
y_train = y.loc[train_mask]
y_test = y.loc[~train_mask]

print(f"Rows in train set: {y_train.shape[0]}, Rows in test set: {y_test.shape[0]}")
print(f"Mean of y_train: {y_train.mean()}, Mean of y_test: {y_test.mean()}")

Shape of x: (545779, 80)
Mean of y: 0.14068331687367963
Rows in train set: 262992, Rows in test set: 282787
Mean of y_train: 0.1319165601995498, Mean of y_test: 0.14883640337073487


In [2]:
df = pd.read_parquet("data/processed/model_table.parquet")

x = df.drop(columns=["id", "default_flag", "issue_year", "issue_date", "funded_amnt", "annual_inc"])
x = pd.get_dummies(
    x,
    columns=["home_ownership", "verification_status", "purpose"],
    drop_first=True,
    dtype=int,
)

prune_cols = [
    "num_sats", "num_rev_tl_bal_gt_0", "total_acc", "tot_hi_cred_lim",
    "total_il_high_credit_limit", "total_bc_limit", "bc_util",
    "ever_mths_since_last_major_derog", "ever_mths_since_recent_bc_dlq",
    "ever_mths_since_recent_revol_delinq", "ever_mths_since_last_delinq",
    "ever_mths_since_last_record", "pct_tl_nvr_dlq", "mo_sin_old_rev_tl_op",
    "revol_bal", "total_rev_hi_lim", "pub_rec", "num_op_rev_tl", "num_rev_accts",
    "mo_sin_rcnt_rev_tl_op", "mths_since_last_record", "mths_since_recent_revol_delinq",
    "num_tl_90g_dpd_24m",
]
x = x.drop(columns=prune_cols)

coefs = pd.read_csv("outputs/lr_coefficients.csv")
mask_2015 = df["issue_year"] == 2015
X = x.loc[mask_2015].copy()
X = X[coefs["feature"]]

print(f"2015 rows: {len(X)}")
print(f"Feature columns: {X.shape[1]}")
assert X.columns.tolist() == coefs["feature"].tolist()
print(f"Feature order matches coefficients: {len(X.columns)} features")

z = (X - coefs["mean"].values) / coefs["scale"].values
contrib = z * coefs["coef"].values

import json
with open("outputs/metrics.json") as f:
    intercept = json.load(f)["lr_57"]["intercept"]
pd_check = 1 / (1 + np.exp(-(intercept + contrib.sum(axis=1))))

scored_test = pd.read_parquet("data/processed/scored_test.parquet")
pd_comparison = pd.DataFrame({"id": df.loc[mask_2015, "id"].to_numpy(), "pd_check": pd_check.to_numpy()}).merge(
    scored_test[["id", "pd_lr"]],
    on="id",
    how="left",
    validate="one_to_one",
)
pd_comparison = pd_comparison.set_index("id").loc[df.loc[mask_2015, "id"]].reset_index()
max_abs_diff = (pd_comparison["pd_check"] - pd_comparison["pd_lr"]).abs().max()
print(max_abs_diff)

declined_mask = pd_comparison["pd_lr"].ge(0.18)
declined = pd_comparison.loc[declined_mask].copy()
top3 = contrib.loc[declined_mask.to_numpy()].apply(
    lambda row: row.nlargest(3).index.tolist(),
    axis=1,
)

print(f"Declined applicants: {len(declined)} ({len(declined) / len(pd_comparison)})")
print("Top 10 reason features:")
print(top3.explode().value_counts().head(10))

2015 rows: 282787
Feature columns: 57
Feature order matches coefficients: 57 features
2.55351295663786e-15
Declined applicants: 71287 (0.25208726002256115)
Top 10 reason features:
fico                    38457
acc_open_past_24mths    33596
loan_to_income          24324
dti                     21226
log_annual_inc          20509
num_tl_op_past_12m      13971
home_ownership_RENT     11982
percent_bc_gt_75        11145
emp_length_missing       9757
inq_last_6mths           6816
Name: count, dtype: int64


In [3]:
ineligible = [
    c for c in contrib.columns
    if c.startswith(("home_ownership_", "purpose_", "verification_status_"))
]
contrib_elig = contrib.drop(columns=ineligible)

top3_2 = contrib_elig.loc[declined_mask.to_numpy()].apply(
    lambda row: row[row > 0].nlargest(3).index.tolist(),
    axis=1,
)

print("Top 14 reason features:")
print(top3_2.explode().value_counts().head(14))
print(contrib_elig.shape[1])

Top 14 reason features:
fico                     41163
acc_open_past_24mths     34457
loan_to_income           25784
dti                      23650
log_annual_inc           23505
num_tl_op_past_12m       17048
percent_bc_gt_75         13143
emp_length_missing        9767
inq_last_6mths            8736
delinq_2yrs               4210
num_actv_rev_tl           3512
mths_since_recent_inq     1605
open_acc                  1394
num_il_tl                 1372
Name: count, dtype: int64
43


In [5]:
reason_names = {
    "fico": "credit score is too low",
    "acc_open_past_24mths": "too many accounts opened in the past two years",
    "loan_to_income": "ratio of loan amount to annual income is too high",
    "dti": "monthly debt payments are too high relative to income",
    "log_annual_inc": "annual income is too low",
    "num_tl_op_past_12m": "too many accounts opened in the past year",
    "percent_bc_gt_75": "percentage of accounts near max balance is too high",
    "emp_length_missing": "length of employment left blank",
    "inq_last_6mths": "number of recent lender credit pulls is too high",
    "delinq_2yrs": "number of late payments is too high",
    "num_actv_rev_tl": "number of revolving accounts that carry a balance is too high",
    "mths_since_recent_inq": "the most recent credit inquiry was too recent",
    "open_acc": "number of credit accounts currently open is too low",
    "num_il_tl": "number of installment loans is too low",
    "loan_amnt": "amount of loan requested was too high",
    "bc_open_to_buy": "unused credit amount is too low",
    "num_bc_sats": "number of bank accounts in good standing is too high",
    "collections_12_mths_ex_med": "number of recent non-medical collections is too high",
    "num_bc_tl": "number of bank card accoutns ever opened is too low",
    "num_tl_30pd": "number of current overdue accounts is too high",
    "total_bal_ex_mort": "amount of total debt excluding mortgage is too high"
}

def format_value(feature, value):
    if feature == "fico":
        return f"{value:.0f}"
    elif feature == "log_annual_inc":
        return f"${np.exp(value):,.0f}"
    elif feature in ["loan_to_income"]:
        return f"{value:.0%}"
    elif feature == "emp_length_missing":
        return "not provided"
    elif feature == "percent_bc_gt_75":
        return f"{value:.0f}%"
    elif feature == "dti":
            return f"{value:.0f}%"
    elif feature == "mths_since_recent_inq":
            return f"{value} months"
    else:
        return f"{value:.0f}"

mapped_rows = top3_2[top3_2.apply(lambda factors: all(feature in reason_names for feature in factors))]
sample_ids = mapped_rows.sample(25, random_state=0).index
pd_by_index = pd.Series(pd_comparison["pd_lr"].to_numpy(), index=X.index)
rows = []

for idx in sample_ids:
    factors = top3_2.loc[idx]
    rows.append({
        "row_idx": idx,
        "pd": pd_by_index.loc[idx],
        "factor_1": reason_names[factors[0]],
        "value_1": format_value(factors[0], X.loc[idx, factors[0]]),
        "factor_2": reason_names[factors[1]],
        "value_2": format_value(factors[1], X.loc[idx, factors[1]]),
        "factor_3": reason_names[factors[2]],
        "value_3": format_value(factors[2], X.loc[idx, factors[2]]),
    })

applicant_reasons = pd.DataFrame(rows)
pd.set_option("display.max_colwidth", None)
print(applicant_reasons.to_string())

    row_idx        pd                                               factor_1       value_1                                                       factor_2       value_2                                                       factor_3  value_3
0    386512  0.348683                               annual income is too low       $15,000                                length of employment left blank  not provided              ratio of loan amount to annual income is too high      40%
1    228298  0.290603                    number of late payments is too high             4                                        credit score is too low           667                                       annual income is too low  $36,000
2    149678  0.239264         too many accounts opened in the past two years             8          monthly debt payments are too high relative to income           32%                      too many accounts opened in the past year        5
3    394390  0.242784      ratio of loan